<a href="https://colab.research.google.com/github/hithamab/presentation/blob/master/Copy_of_Masarat_S3A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""DeepSeek RAG Customer Support Bot

Complete implementation using DeepSeek API for Arabic/English customer support.
Uses RAG with PDF documents for accurate, context-aware responses.
"""

# 1. INSTALL REQUIRED PACKAGES
!pip install -q --upgrade langchain langchain-deepseek langchain-community \
  langchain-chroma langchain-huggingface langchain-text-splitters \
  chromadb python-dotenv sentence-transformers gradio pypdf tiktoken

import os
from dotenv import load_dotenv
from google.colab import drive, userdata

# 2. LOAD ENVIRONMENT VARIABLES
load_dotenv()

# 3. SET DEEPSEEK API KEY
# Get API key from Google Colab secrets (Tools → Secrets)
# Add a secret with key: DEEPSEEK_API_KEY and your API as value
os.environ["DEEPSEEK_API_KEY"] = userdata.get('DEEPSEEK_API_KEY')

# 4. MOUNT GOOGLE DRIVE (for persistent storage)
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.2/328.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 477.4/477.4 kB 36.0 MB/s eta 0:00:

In [ ]:
import os

# ==============================================
# CONFIGURATION SECTION
# ==============================================

# 1. PDF DOCUMENT PATH
# Path to your Arabic/English FAQ PDF in Google Drive
PDF_PATH = "/content/drive/MyDrive/customer_support_arabic_full.pdf"

# 2. CHROMA VECTOR DATABASE PATH
# Where to store/load the vector embeddings
CHROMA_DIR = "/content/drive/MyDrive/chroma_customer_support_deepseek"

# 3. CHECK IF PDF EXISTS
def check_pdf_exists(path):
    """Verify the source PDF document exists."""
    if os.path.exists(path):
        print(f"✅ Source PDF found: {path}")
        print(f"   File size: {os.path.getsize(path) / 1024:.2f} KB")
        return True
    else:
        print(f"❌ PDF not found at: {path}")
        print("   Please ensure the file exists in Google Drive")
        return False

# Run the check
check_pdf_exists(PDF_PATH)

✅ Source PDF found: /content/drive/MyDrive/customer_support_arabic_full.pdf
   File size: 107.98 KB


True

In [ ]:
# ==============================================
# DOCUMENT PROCESSING PIPELINE
# ==============================================

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import tiktoken

def load_and_process_documents(pdf_path):
    """
    Load PDF and split into chunks for embedding.

    Args:
        pdf_path: Path to the PDF file

    Returns:
        chunks: List of document chunks
        total_pages: Number of pages in PDF
    """

    print("📄 Loading PDF document...")

    # 1. Load PDF
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    total_pages = len(docs)
    print(f"   Loaded {total_pages} pages from PDF")

    # 2. Configure text splitter
    # Optimized for Arabic/English mixed content
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,           # Adjust based on your content
        chunk_overlap=150,        # Maintain context between chunks
        length_function=len,
        separators=["\n\n", "\n", "۔", "۔ ", ". ", " ", ""]  # Arabic/English separators
    )

    # 3. Split documents
    chunks = splitter.split_documents(docs)
    print(f"   Created {len(chunks)} text chunks")
    print(f"   Average chunk length: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")

    # 4. Show sample chunk (first 200 chars)
    if chunks:
        print(f"   Sample chunk: {chunks[0].page_content[:200]}...")

    return chunks, total_pages

# Process the documents
chunks, total_pages = load_and_process_documents(PDF_PATH)

📄 Loading PDF document...
   Loaded 4 pages from PDF
   Created 9 text chunks
   Average chunk length: 587 chars
   Sample chunk: دﻟﯾل ﺧدﻣﺔ اﻟﻌﻣﻼء  –  ﺷرﻛﺔ  ﺳﻣﺎرت  ﺗك  ﻟﻺﻟﻛﺗروﻧﯾﺎت  
اﻟﻣﻘدﻣﺔ 
ﺗﮭدف ﺷرﻛﺔ ﺳﻣﺎرت  ﺗك  إﻟﻰ  ﺗﻘدﯾم  ﺗﺟرﺑﺔ  ﺷراء  آﻣﻧﺔ  وﺳﮭﻠﺔ  ﻟﺟﻣﯾﻊ  ﻋﻣﻼﺋﮭﺎ  داﺧل  وﺧﺎرج  اﻟدوﻟﺔ.  
 
ﺗﻌﻣل
 
ﺧدﻣﺔ
 
اﻟﻌﻣﻼء
 
ﻋﻠﻰ
 
ﻣدار
 
اﻷﺳﺑ...


In [ ]:
# ==============================================
# VECTOR DATABASE (ChromaDB)
# ==============================================

from langchain_chroma import Chroma
import hashlib

def setup_vector_database(chunks, chroma_dir, force_rebuild=False):
    """
    Create or load Chroma vector database.

    Args:
        chunks: Document chunks to embed
        chroma_dir: Directory for Chroma persistence
        force_rebuild: Force rebuilding the database

    Returns:
        vectordb: Chroma vector database instance
        retriever: Document retriever
    """

    # 1. Choose embedding model
    # multilingual-e5-small works well for Arabic/English
    print("🔤 Loading embedding model...")
    embeddings = HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small",
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )

    # 2. Check if database exists
    db_exists = os.path.exists(chroma_dir) and not force_rebuild

    if db_exists:
        print(f"📂 Loading existing vector database from: {chroma_dir}")
        vectordb = Chroma(
            persist_directory=chroma_dir,
            embedding_function=embeddings
        )
        print(f"   Database contains {vectordb._collection.count()} vectors")
    else:
        print("🛠️ Building new vector database...")
        if force_rebuild and os.path.exists(chroma_dir):
            import shutil
            shutil.rmtree(chroma_dir)
            print("   Removed old database")

        # Create new vector store
        vectordb = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory=chroma_dir,
            collection_metadata={"hnsw:space": "cosine"}  # Cosine similarity
        )
        print(f"   Created database with {len(chunks)} vectors")
        print(f"   Saved to: {chroma_dir}")

    # 3. Create retriever with MMR for diverse results
    retriever = vectordb.as_retriever(
        search_type="mmr",  # Maximal Marginal Relevance
        search_kwargs={
            "k": 6,           # Number of chunks to retrieve
            "fetch_k": 20,    # Initial pool for MMR
            "lambda_mult": 0.7  # Diversity vs relevance balance
        }
    )

    print("✅ Vector database setup complete")
    return vectordb, retriever

# Setup the vector database
vectordb, retriever = setup_vector_database(chunks, CHROMA_DIR)

🔤 Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

📂 Loading existing vector database from: /content/drive/MyDrive/chroma_customer_support_deepseek
   Database contains 9 vectors
✅ Vector database setup complete


In [ ]:
# ==============================================
# DEEPSEEK LLM SETUP
# ==============================================

from langchain_deepseek import ChatDeepSeek
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

def setup_deepseek_llm():
    """
    Configure the DeepSeek language model.

    Returns:
        llm: Configured DeepSeek LLM instance
    """

    print("🤖 Setting up DeepSeek LLM...")

    # Available DeepSeek models:
    # 1. "deepseek-chat" - General purpose chat
    # 2. "deepseek-reasoner" - Better for complex reasoning

    llm = ChatDeepSeek(
        model="deepseek-chat",  # Use "deepseek-reasoner" for complex queries
        temperature=0.2,        # Lower = more focused, Higher = more creative
        max_tokens=1000,        # Maximum response length
        timeout=30,             # Timeout in seconds
        max_retries=2,          # Retry on failure
        api_key=os.environ.get("DEEPSEEK_API_KEY")
    )

    # Test the connection
    try:
        test_response = llm.invoke("Hello")
        print(f"✅ DeepSeek connection successful")
        print(f"   Model: deepseek-chat")
        print(f"   Temperature: 0.2")
        print(f"   Max tokens: 1000")
    except Exception as e:
        print(f"❌ DeepSeek connection failed: {str(e)}")
        print("   Please check your API key in Colab Secrets")
        raise

    return llm

# Initialize DeepSeek
llm = setup_deepseek_llm()

🤖 Setting up DeepSeek LLM...
✅ DeepSeek connection successful
   Model: deepseek-chat
   Temperature: 0.2
   Max tokens: 1000


In [ ]:
# ==============================================
# RAG CHAIN CONSTRUCTION
# ==============================================

def format_docs(docs):
    """
    Format retrieved documents for the prompt.

    Args:
        docs: List of retrieved documents

    Returns:
        Formatted context string
    """
    return "\n\n".join([f"[Source {i+1}]: {d.page_content}"
                       for i, d in enumerate(docs)])

# System prompt in Arabic (adapt for your needs)
SYSTEM_PROMPT_ARABIC = """
أنت مساعد ذكي لخدمة عملاء شركة إلكترونيات.
مهمتك الأساسية هي مساعدة المستخدمين بناءً على المعلومات الموجودة في قاعدة المعرفة فقط.

## 📋 قواعد الرد:
1. استخدم **فقط** المعلومات الموجودة في النصوص المقدمة أدناه.
2. إذا لم تكن الإجابة موجودة في النصوص، قل: "عذرًا، لم أجد إجابة محددة في معلوماتنا. يُرجى التواصل مع فريق الدعم البشري للحصول على مساعدة إضافية."
3. اجعل الردود **قصيرة وواضحة ومباشرة**.
4. حافظ على نفس لغة سؤال المستخدم (عربي أو إنجليزي).
5. إذا كان السؤال يحتوي على أخطاء إملائية أو غير واضح، حاول تخمين القصد وأجب بأفضل ما يمكن بناءً على المعلومات المتاحة.

## النصوص المرجعية:
{context}
"""

# English version of the prompt
SYSTEM_PROMPT_ENGLISH = """
You are an intelligent customer support assistant for an electronics company.
Your primary task is to help users based ONLY on the information in the knowledge base.

## 📋 Response Rules:
1. Use **ONLY** the information found in the texts provided below.
2. If the answer is not in the texts, say: "Sorry, I couldn't find a specific answer in our information. Please contact our human support team for further assistance."
3. Keep responses **short, clear, and direct**.
4. Maintain the same language as the user's question (Arabic or English).
5. If the question has typos or is unclear, try to guess the intent and answer based on the available information.

## Reference Texts:
{context}
"""

def create_rag_chain(llm, retriever, language="arabic"):
    """
    Create the complete RAG pipeline.

    Args:
        llm: Language model instance
        retriever: Document retriever
        language: "arabic" or "english" for prompt language

    Returns:
        rag_chain: Complete RAG pipeline
    """

    print(f"🔗 Creating RAG chain ({language} prompt)...")

    # Select prompt based on language
    system_prompt = SYSTEM_PROMPT_ARABIC if language == "arabic" else SYSTEM_PROMPT_ENGLISH

    # Create prompt template
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{question}")
    ])

    # Build RAG chain
    rag_chain = (
        RunnableParallel({
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        })
        | prompt
        | llm
    )

    print("✅ RAG chain created successfully")
    return rag_chain

# Create the RAG chain
rag_chain = create_rag_chain(llm, retriever, language="arabic")

🔗 Creating RAG chain (arabic prompt)...
✅ RAG chain created successfully


In [ ]:
# ==============================================
# GRADIO CHAT INTERFACE
# ==============================================

import gradio as gr
import time
from typing import List, Dict, Tuple

class CustomerSupportBot:
    """Main chatbot class handling conversations."""

    def __init__(self, rag_chain):
        self.rag_chain = rag_chain
        self.conversation_history = []

    def respond(self, message: str, chat_history: List[Dict]) -> Tuple[List[Dict], str]:
        """
        Process user message and generate response.

        Args:
            message: User input
            chat_history: Previous conversation

        Returns:
            Updated chat history and empty message box
        """

        # Validate input
        if not message or not message.strip():
            return chat_history, ""

        print(f"👤 User: {message[:100]}...")
        start_time = time.time()

        try:
            # Get response from RAG chain
            response = self.rag_chain.invoke(message)
            bot_reply = response.content

            # Calculate response time
            response_time = time.time() - start_time

            print(f"🤖 Bot: {bot_reply[:100]}...")
            print(f"⏱️ Response time: {response_time:.2f} seconds")

        except Exception as e:
            error_msg = f"⚠️ Sorry, an error occurred. Please try again. Error: {str(e)}"
            print(f"❌ Error: {str(e)}")
            bot_reply = error_msg

        # Update conversation history
        if chat_history is None:
            chat_history = []

        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": bot_reply})

        # Limit history to last 20 messages
        if len(chat_history) > 20:
            chat_history = chat_history[-20:]

        return chat_history, ""

def create_gradio_interface(bot):
    """
    Create the Gradio web interface.

    Args:
        bot: CustomerSupportBot instance

    Returns:
        demo: Gradio interface
    """

    print("🎨 Creating Gradio interface...")

    # Define theme and css separately for Gradio 6.0 compatibility
    custom_theme = gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="gray",
        font=[gr.themes.GoogleFont("Noto Sans Arabic"), "Arial", "sans-serif"]
    )
    custom_css = ".gradio-container {max-width: 800px; margin: auto;}"

    with gr.Blocks() as demo:

        # Header
        gr.Markdown("""
        # 🤖 مساعد الدعم الفني الذكي
        ### المدعوم بـ DeepSeek AI ونظام RAG

        **كيفية الاستخدام:**
        1. اكتب سؤالك باللغة العربية أو الإنجليزية
        2. اضغط Enter أو زر الإرسال
        3. سيجيب المساعد بناءً على معلومات قاعدة المعرفة

        **ملاحظة:** الإجابات مبنية فقط على المعلومات الموجودة في ملفات الدعم.
        """)

        # Chatbot display
        chatbot = gr.Chatbot(
            label="محادثة الدعم الفني",
            height=500,
            show_label=True,
            avatar_images=(None, "🤖")
        )

        # Input area
        msg = gr.Textbox(
            placeholder="اكتب سؤالك هنا بالعربية أو الإنجليزية...",
            label="رسالتك",
            lines=2,
            max_lines=4,
            show_label=True
        )

        # Buttons
        with gr.Row():
            submit_btn = gr.Button("إرسال 🚀", variant="primary")
            clear_btn = gr.Button("مسح المحادثة 🗑️", variant="secondary")

        # Statistics display
        with gr.Row():
            stats = gr.Markdown("**الإحصائيات:** جاهز للبدء")

        # Footer
        gr.Markdown("""
        ---
        **معلومات النظام:**
        - النموذج: DeepSeek-Chat
        - قاعدة المعرفة: وثائق الدعم الفني
        - التخزين: قاعدة بيانات ناقلات في Google Drive
        """)

        # Event handlers
        def update_stats():
            """Update statistics display."""
            return f"**الإحصائيات:** {len(bot.conversation_history)} رسائل في هذه الجلسة"

        # Connect submit action
        submit_event = msg.submit(
            fn=bot.respond,
            inputs=[msg, chatbot],
            outputs=[chatbot, msg]
        )

        # Connect button click
        submit_click = submit_btn.click(
            fn=bot.respond,
            inputs=[msg, chatbot],
            outputs=[chatbot, msg]
        )

        # Connect clear button
        def clear_chat():
            bot.conversation_history = []
            return [], ""

        clear_btn.click(
            fn=clear_chat,
            inputs=None,
            outputs=[chatbot, msg]
        )

        # Update stats on any interaction
        for event in [submit_event, submit_click]:
            event.then(update_stats, None, stats)

    print("✅ Gradio interface created")
    # Pass theme and css to launch() as recommended by Gradio 6.0 warning
    return demo, custom_theme, custom_css

# Initialize bot and create interface
bot = CustomerSupportBot(rag_chain)
demo, custom_theme, custom_css = create_gradio_interface(bot)

# The Gradio app would typically be launched here:
# demo.launch(theme=custom_theme, css=custom_css)

🎨 Creating Gradio interface...
✅ Gradio interface created


In [ ]:
# ==============================================
# LAUNCH THE APPLICATION
# ==============================================

def launch_application():
    """
    Launch the Gradio application with sharing options.
    """

    print("\n" + "="*50)
    print("🚀 LAUNCHING CUSTOMER SUPPORT BOT")
    print("="*50)

    # Display configuration summary
    print("\n📊 CONFIGURATION SUMMARY:")
    print(f"   • PDF Source: {PDF_PATH}")
    print(f"   • Vector DB: {CHROMA_DIR}")
    print(f"   • LLM: DeepSeek-Chat")
    print(f"   • Embeddings: multilingual-e5-small")
    print(f"   • Chunks: {len(chunks)}")
    print(f"   • Pages: {total_pages}")

    print("\n🌐 Starting Gradio server...")
    print("   Note: The public URL will appear below")
    print("   Press Ctrl+C in the cell to stop the server")
    print("="*50 + "\n")

    # Launch with public sharing
    try:
        demo.launch(
            share=True,           # Create public link
            debug=False,          # Set to True for debugging
            server_name="0.0.0.0",
            server_port=7860
        )
    except KeyboardInterrupt:
        print("\n⏹️ Server stopped by user")
    except Exception as e:
        print(f"\n❌ Failed to launch: {str(e)}")

# Launch the application
launch_application()


🚀 LAUNCHING CUSTOMER SUPPORT BOT

📊 CONFIGURATION SUMMARY:
   • PDF Source: /content/drive/MyDrive/customer_support_arabic_full.pdf
   • Vector DB: /content/drive/MyDrive/chroma_customer_support_deepseek
   • LLM: DeepSeek-Chat
   • Embeddings: multilingual-e5-small
   • Chunks: 9
   • Pages: 4

🌐 Starting Gradio server...
   Note: The public URL will appear below
   Press Ctrl+C in the cell to stop the server

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://af3f946c89691bd9b7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
